In [1]:
import pandas as pd
from datasets import load_dataset


In [4]:
sample_size = 30
seed = 42
dataset_configs = {
    'fineweb': {"path": "HuggingFaceFW/fineweb", "name": "sample-10BT", "split": "train"},
    'finepdfs': {"path": "HuggingFaceFW/finepdfs", "name": "eng_Latn", "split": "train"},
    'C4': {"path": "allenai/c4", "name": "en", "split": "train"},
    'RedPajama': {"path": "krisbailey/RedPajama-Data-V2-1B", "name": None, "split": "train"}
}

In [ ]:
for name, config in dataset_configs.items():
    # load dataset with predefined parameters
    kwargs = {"path": config["path"], "split": config["split"], "streaming": True}
    if config["name"]:
        kwargs["name"] = config["name"]

    ds = load_dataset(**kwargs)

    # take random sample (of first 10k documents)
    sampled_stream = ds.shuffle(seed=seed, buffer_size=10000).take(sample_size)
    samples_list = list(sampled_stream)
    df = pd.DataFrame(samples_list)

    # It's not obvious which columns are useful as id's so we try different ones
    if 'id' in df.columns:
        df['document_index'] = df['id']
        print(f"{name} has 'id' column.")
    elif 'doc_id' in df.columns:
        df['document_index'] = df['doc_id']
        print(f"{name} has 'doc_id' column.")
    elif 'url' in df.columns:
        df['document_index'] = df['url']
        print(f"{name} has 'url' column.")
    else:
        # error if no id-like column is found
        raise ValueError(f"{name} does not have an 'id', 'doc_id', or 'url' column to use as a document index.")
        

    # Save
    filename = f"{name}_dataset.json"
    df.to_json(filename, orient="records", indent=2)

print("\nDone")